In [1]:
"""
Reinforcement Learning — Two Gymnasium Examples
================================================

Example 1: FrozenLake-v1  (discrete state/action)
    Environment : 4x4 grid, reach goal without falling in holes
    Algorithms  : Q-learning and SARSA  (tabular — small state space)
    Why tabular : only 16 states x 4 actions = 64 Q-table cells

Example 2: CartPole-v1  (continuous state)
    Environment : balance a pole on a moving cart
    Algorithms  : DQN and REINFORCE  (neural network — continuous state)
    Why network : state = 4 continuous numbers (position, velocity, angle, angular velocity)
                  too many states for a table

Key point:
    The ONLY difference from our hand-built examples is the environment.
    The agent code (Q-table update, network architecture, training loop) is identical.
    Gymnasium just replaces our hand-written env_step() and env_reset().

Gymnasium interface (same for every environment):
    state, info            = env.reset()
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
"""

'\nReinforcement Learning — Two Gymnasium Examples\n================================================\n\nExample 1: FrozenLake-v1  (discrete state/action)\n    Environment : 4x4 grid, reach goal without falling in holes\n    Algorithms  : Q-learning and SARSA  (tabular — small state space)\n    Why tabular : only 16 states x 4 actions = 64 Q-table cells\n\nExample 2: CartPole-v1  (continuous state)\n    Environment : balance a pole on a moving cart\n    Algorithms  : DQN and REINFORCE  (neural network — continuous state)\n    Why network : state = 4 continuous numbers (position, velocity, angle, angular velocity)\n                  too many states for a table\n\nKey point:\n    The ONLY difference from our hand-built examples is the environment.\n    The agent code (Q-table update, network architecture, training loop) is identical.\n    Gymnasium just replaces our hand-written env_step() and env_reset().\n\nGymnasium interface (same for every environment):\n    state, info            = 

In [ ]:
import numpy as np
import random
from collections import deque
import gymnasium as gym

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not found — DQN and REINFORCE will be skipped.")

# ═════════════════════════════════════════════════════════════════════════════
# EXAMPLE 1 — FROZENLAKE  (Q-learning and SARSA)
# ═════════════════════════════════════════════════════════════════════════════
"""
FrozenLake-v1 grid (4x4):
    S F F F        S = Start
    F H F H        F = Frozen (safe)
    F F F H        H = Hole   (fall in = episode ends, reward 0)
    H F F G        G = Goal   (reward +1)

States:  16 (numbered 0-15, row by row)
Actions: 4  (0=LEFT, 1=DOWN, 2=RIGHT, 3=UP)
Rewards: +1 at goal, 0 everywhere else
Note:    is_slippery=False makes moves deterministic (easier to learn)
"""

def run_frozenlake_qlearning(n_episodes=5000, alpha=0.5, gamma=0.95,
                              epsilon_start=1.0, epsilon_min=0.01,
                              epsilon_decay=0.999, verbose=True):
    """
    Q-learning on FrozenLake-v1.

    Same algorithm as our 2x2 grid — only the environment changes.
    Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',.) - Q(s,a)]
    """
    print("\n" + "="*60)
    print("EXAMPLE 1A: Q-Learning on FrozenLake-v1")
    print("="*60)

    # ── Create the Gymnasium environment ──────────────────────────────────
    env = gym.make("FrozenLake-v1", is_slippery=False)

    n_states  = env.observation_space.n    # 16
    n_actions = env.action_space.n         # 4

    print(f"States: {n_states}, Actions: {n_actions}")
    print(f"Q-table size: {n_states} x {n_actions} = {n_states * n_actions} cells")

    # ── Q-table: 16 states x 4 actions, all zeros ─────────────────────────
    Q = np.zeros((n_states, n_actions))
    epsilon = epsilon_start

    rewards_per_episode = []
    success_count = 0

    for episode in range(n_episodes):

        # Gymnasium reset() returns (state, info) — unpack info with _
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            # ── Action selection: epsilon-greedy ───────────────────────────
            if np.random.random() < epsilon:
                action = env.action_space.sample()       # EXPLORE: random
            else:
                action = int(np.argmax(Q[state]))        # EXPLOIT: best Q

            # ── Gymnasium step() returns 5 values ─────────────────────────
            # (compare to our hand-built: next_state, reward, done = env_step())
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated               # combine both flags

            total_reward += reward

            # ── Q-learning update (identical to hand-built version) ────────
            best_next = 0.0 if done else np.max(Q[next_state])
            td_error  = reward + gamma * best_next - Q[state, action]
            Q[state, action] += alpha * td_error

            state = next_state

        rewards_per_episode.append(total_reward)
        if total_reward > 0:
            success_count += 1

        # decay epsilon after each episode
        # this is to ensure that the agent explores more in the beginning and exploits more towards the end
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    env.close()

    # ── Results ────────────────────────────────────────────────────────────
    if verbose:
        action_names = ["LEFT", "DOWN", "RIGHT", "UP"]

        print(f"\nTraining complete ({n_episodes} episodes)")
        print(f"Success rate (last 100 episodes): "
              f"{sum(rewards_per_episode[-100:]):.0f}%")

        print("\nLearned policy (best action per cell, 4x4 grid):")
        policy = [action_names[np.argmax(Q[s])] for s in range(n_states)]
        for row in range(4):
            print("  " + "  ".join(f"{policy[row*4+col]:>5}"
                                    for col in range(4)))

        print("\nQ-table (showing max Q per state, 4x4 grid):")
        for row in range(4):
            vals = [f"{np.max(Q[row*4+col]):5.2f}" for col in range(4)]
            print("  " + "  ".join(vals))

        print("\nKey Gymnasium difference from hand-built:")
        print("  env.reset()  returns (state, info)  — not just state")
        print("  env.step()   returns (state, reward, terminated, truncated, info)")
        print("  done = terminated or truncated  — combine both flags")
        print("  Everything else (Q-table, update formula) is IDENTICAL")

    return Q

In [5]:
def run_frozenlake_sarsa(n_episodes=5000, alpha=0.5, gamma=0.95,
                          epsilon_start=1.0, epsilon_min=0.01,
                          epsilon_decay=0.999, verbose=True):
    """
    SARSA on FrozenLake-v1.

    Same difference from Q-learning as before:
    Uses Q(s',a') — the actual next action — instead of max Q(s',.)
    """
    print("\n" + "="*60)
    print("EXAMPLE 1B: SARSA on FrozenLake-v1")
    print("="*60)

    env = gym.make("FrozenLake-v1", is_slippery=False)

    n_states  = env.observation_space.n
    n_actions = env.action_space.n

    print(f"States: {n_states}, Actions: {n_actions}")

    Q = np.zeros((n_states, n_actions))
    epsilon = epsilon_start

    rewards_per_episode = []

    def choose_action(state):
        if np.random.random() < epsilon:
            return env.action_space.sample()
        return int(np.argmax(Q[state]))

    for episode in range(n_episodes):
        state, _ = env.reset()
        action   = choose_action(state)          # choose FIRST action upfront
        done = False
        total_reward = 0

        while not done:
            # take current action
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            # choose NEXT action BEFORE updating (key SARSA requirement)
            if done:
                next_action = 0
            else:
                next_action = choose_action(next_state)

            # SARSA update: uses Q of ACTUAL next action
            next_q    = 0.0 if done else Q[next_state, next_action]
            td_error  = reward + gamma * next_q - Q[state, action]
            Q[state, action] += alpha * td_error

            # carry chosen action forward
            state  = next_state
            action = next_action

        rewards_per_episode.append(total_reward)
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    env.close()

    if verbose:
        action_names = ["LEFT", "DOWN", "RIGHT", "UP"]

        print(f"\nTraining complete ({n_episodes} episodes)")
        print(f"Success rate (last 100 episodes): "
              f"{sum(rewards_per_episode[-100:]):.0f}%")

        print("\nLearned policy (best action per cell, 4x4 grid):")
        policy = [action_names[np.argmax(Q[s])] for s in range(n_states)]
        for row in range(4):
            print("  " + "  ".join(f"{policy[row*4+col]:>5}"
                                    for col in range(4)))

        print("\nKey difference from Q-learning:")
        print("  next_action = choose_action(next_state)  <- epsilon-greedy picks a'")
        print("  next_q      = Q[next_state, next_action] <- uses a', not max")
        print("  Conservative: accounts for its own exploratory mistakes")

    return Q


In [6]:
# ═════════════════════════════════════════════════════════════════════════════
# EXAMPLE 2 — CARTPOLE  (DQN and REINFORCE)
# ═════════════════════════════════════════════════════════════════════════════
"""
CartPole-v1:
    Goal:    keep a pole balanced on a moving cart for as long as possible
    State:   4 continuous numbers
               [cart position, cart velocity, pole angle, pole angular velocity]
    Actions: 2 discrete (0=push left, 1=push right)
    Reward:  +1 for every step the pole stays upright
    Done:    pole angle > 12 degrees OR cart off screen OR 500 steps reached

Why neural network:
    State is 4 continuous numbers — infinite possible states
    Cannot build a Q-table (would need infinite rows)
    Neural network takes the 4 numbers directly and estimates Q-values
    No one-hot encoding needed (state is already numbers)
"""

def run_cartpole_dqn(n_episodes=300, gamma=0.99, lr=1e-3,
                     epsilon_start=1.0, epsilon_min=0.05,
                     epsilon_decay=0.995, batch_size=64,
                     target_update_every=100, verbose=True):
    """
    DQN on CartPole-v1.

    Same algorithm as our 2x2 DQN — only the environment changes.
    State is 4 continuous numbers instead of one-hot encoded label.
    Network takes those 4 numbers directly (no encoding needed).
    """
    if not TORCH_AVAILABLE:
        print("\nDQN skipped — PyTorch not installed.")
        return None

    print("\n" + "="*60)
    print("EXAMPLE 2A: DQN on CartPole-v1")
    print("="*60)

    env = gym.make("CartPole-v1")

    n_states  = env.observation_space.shape[0]   # 4 continuous numbers
    n_actions = env.action_space.n               # 2 (left or right)

    print(f"State size: {n_states} continuous numbers")
    print(f"Actions: {n_actions} (0=left, 1=right)")
    print("No Q-table possible — using neural network")

    # ── Q-Network: 4 inputs -> 2 Q-values ─────────────────────────────────
    class QNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_states, 64), nn.ReLU(),   # 4 -> 64
                nn.Linear(64, 64),       nn.ReLU(),   # 64 -> 64
                nn.Linear(64, n_actions)              # 64 -> 2 Q-values
            )                                         # NO softmax — outputs values
        def forward(self, x):
            return self.net(x)

    main_net   = QNetwork()
    target_net = QNetwork()
    target_net.load_state_dict(main_net.state_dict())

    optimizer  = torch.optim.Adam(main_net.parameters(), lr=lr)
    loss_fn    = nn.MSELoss()
    buffer     = deque(maxlen=10000)
    epsilon    = epsilon_start
    step_count = 0

    def choose_action(state):
        if np.random.random() < epsilon:
            return env.action_space.sample()                    # EXPLORE
        with torch.no_grad():
            q = main_net(torch.FloatTensor(state))              # state is already numbers!
        return int(torch.argmax(q))                             # EXPLOIT

    scores = []

    for episode in range(n_episodes):
        # CartPole reset() returns (state, info) — state is numpy array of 4 floats
        state, _ = env.reset()
        done = False
        score = 0

        while not done:
            action = choose_action(state)

            # step() returns 5 values same as FrozenLake
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            score += reward

            # store experience (state is array of 4 floats, not an index)
            buffer.append((state, action, reward, next_state, done))
            state = next_state
            step_count += 1

            # train when buffer has enough experiences
            if len(buffer) >= batch_size:
                batch = random.sample(buffer, batch_size)
                states, actions, rewards_, next_states, dones = zip(*batch)

                # convert to tensors
                # state arrays stack directly — no encoding needed
                states_t      = torch.FloatTensor(np.array(states))
                next_states_t = torch.FloatTensor(np.array(next_states))
                actions_t     = torch.LongTensor(actions)
                rewards_t     = torch.FloatTensor(rewards_)
                dones_t       = torch.FloatTensor(dones)

                # compute target using frozen target_net
                with torch.no_grad():
                    max_next_q = target_net(next_states_t).max(dim=1)[0]
                    targets    = rewards_t + gamma * max_next_q * (1 - dones_t)

                # compute prediction from main_net
                current_q = main_net(states_t)\
                              .gather(1, actions_t.unsqueeze(1)).squeeze(1)

                loss = loss_fn(current_q, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # refresh target_net periodically
            if step_count % target_update_every == 0:
                target_net.load_state_dict(main_net.state_dict())

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        scores.append(score)

        if verbose and episode % 50 == 0:
            avg = np.mean(scores[-50:]) if len(scores) >= 50 else np.mean(scores)
            print(f"  Episode {episode:3d} | Score {score:5.0f} | "
                  f"Avg(last50) {avg:6.1f} | epsilon {epsilon:.2f}")

    env.close()

    if verbose:
        print(f"\nFinal avg score (last 50 episodes): {np.mean(scores[-50:]):.1f}")
        print("Score of 500 = perfect (pole balanced for max steps)")
        print("\nKey difference from 2x2 grid DQN:")
        print("  State is 4 continuous floats — fed directly to network (no encoding)")
        print("  Network is wider (64 units) to handle more complex state space")
        print("  Buffer is larger (10000) — CartPole episodes are longer")
        print("  Everything else (replay buffer, target network, 7-step loop) IDENTICAL")

    return main_net



In [7]:
def run_cartpole_reinforce(n_episodes=500, gamma=0.99, lr=1e-3,
                            verbose=True):
    """
    REINFORCE on CartPole-v1.

    Same algorithm as our 2x2 REINFORCE — only the environment changes.
    Policy network takes 4 continuous state numbers directly.
    No epsilon-greedy — actions sampled from policy probabilities.
    """
    if not TORCH_AVAILABLE:
        print("\nREINFORCE skipped — PyTorch not installed.")
        return None

    print("\n" + "="*60)
    print("EXAMPLE 2B: REINFORCE on CartPole-v1")
    print("="*60)

    env = gym.make("CartPole-v1")

    n_states  = env.observation_space.shape[0]   # 4
    n_actions = env.action_space.n               # 2

    print(f"State size: {n_states} continuous numbers")
    print(f"Actions: {n_actions} (0=left, 1=right)")

    # ── Policy Network: 4 inputs -> softmax -> 2 action probabilities ──────
    class PolicyNetwork(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_states, 64), nn.ReLU(),   # 4 -> 64
                nn.Linear(64, n_actions),              # 64 -> 2 logits
                nn.Softmax(dim=-1)                     # logits -> probs (sum=1)
            )                                          # SOFTMAX — outputs probabilities
        def forward(self, x):
            return self.net(x)

    policy_net = PolicyNetwork()
    optimizer  = torch.optim.Adam(policy_net.parameters(), lr=lr)

    scores = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        done = False

        # ── Phase 1: play full episode ─────────────────────────────────────
        episode_rewards   = []
        episode_log_probs = []

        while not done:
            # state is array of 4 floats — feed directly to network
            state_t = torch.FloatTensor(state)
            probs   = policy_net(state_t)              # softmax probabilities

            # SAMPLE from probabilities (no epsilon-greedy needed)
            dist   = torch.distributions.Categorical(probs)
            action = dist.sample()                     # weighted random sample
            log_prob = dist.log_prob(action)           # log prob of chosen action

            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            episode_rewards.append(reward)
            episode_log_probs.append(log_prob)
            state = next_state

        score = sum(episode_rewards)
        scores.append(score)

        # ── Phase 2: compute returns G_t (backwards) ──────────────────────
        returns = []
        G = 0.0
        for r in reversed(episode_rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns_t = torch.FloatTensor(returns)

        # normalize returns to reduce variance (common trick for CartPole)
        returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        # ── Phase 3: compute loss and update ──────────────────────────────
        # loss = -sum(G_t * log_prob) for each step
        policy_loss = [-log_p * G_t
                       for log_p, G_t in zip(episode_log_probs, returns_t)]
        loss = torch.stack(policy_loss).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # ── Phase 4: discard episode data (on-policy) ─────────────────────

        if verbose and episode % 50 == 0:
            avg = np.mean(scores[-50:]) if len(scores) >= 50 else np.mean(scores)
            print(f"  Episode {episode:3d} | Score {score:5.0f} | "
                  f"Avg(last50) {avg:6.1f}")

    env.close()

    if verbose:
        print(f"\nFinal avg score (last 50 episodes): {np.mean(scores[-50:]):.1f}")
        print("Score of 500 = perfect (pole balanced for max steps)")
        print("\nKey difference from 2x2 grid REINFORCE:")
        print("  State is 4 continuous floats — fed directly (no one-hot encoding)")
        print("  torch.distributions.Categorical used for cleaner sampling")
        print("  Returns normalized per episode (reduces variance for CartPole)")
        print("  Everything else (softmax output, G_t, no buffer) IDENTICAL")

    return policy_net


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# INFERENCE — deploy any trained Gym model
# ─────────────────────────────────────────────────────────────────────────────
def run_gym_inference(model, env_name, algorithm_name, n_episodes=5):
    """
    Run inference on a trained Gym model.
    No training — just forward pass and pick action.
    """
    print(f"\n--- Inference: {algorithm_name} on {env_name} ---")

    env = gym.make(env_name, is_slippery=False) if "FrozenLake" in env_name else gym.make(env_name)
    scores = []

    for ep in range(n_episodes):
        state, _ = env.reset()
        done = False
        score = 0

        while not done:
            if algorithm_name in ["Q-learning", "SARSA"]:
                # Table lookup — state is an integer index
                action = int(np.argmax(model[state]))

            elif algorithm_name == "DQN":
                # Network: state -> forward pass -> argmax Q-values
                with torch.no_grad():
                    q = model(torch.FloatTensor(state))
                action = int(torch.argmax(q))

            elif algorithm_name == "REINFORCE":
                # Network: state -> softmax -> argmax probabilities
                with torch.no_grad():
                    probs = model(torch.FloatTensor(state))
                action = int(torch.argmax(probs))     # greedy at inference

            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            score += reward

        scores.append(score)

    env.close()
    print(f"  Scores over {n_episodes} episodes: {[int(s) for s in scores]}")
    print(f"  Average score: {np.mean(scores):.1f}")


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPARISON — hand-built vs Gymnasium
# ─────────────────────────────────────────────────────────────────────────────
def print_gym_comparison():
    print("\n" + "="*60)
    print("HAND-BUILT vs GYMNASIUM — what changes")
    print("="*60)
    print("""
    HAND-BUILT ENVIRONMENT:          GYMNASIUM ENVIRONMENT:
    ─────────────────────────────    ──────────────────────────────────
    state = env_reset()              state, info = env.reset()
    next, r, done = env_step(a)      next, r, term, trunc, info = env.step(a)
                                     done = term or trunc
    state is an integer (0,1,2)      FrozenLake: integer (0-15)
                                     CartPole:   array of 4 floats
    encode(state) needed for DQN     FrozenLake: need one-hot encode
                                     CartPole:   NO encode needed (already numbers)

    AGENT CODE (Q-table, network, update formula) — IDENTICAL IN ALL CASES
    The environment is the only thing that changes.
    """)

    print("Algorithm → Best environment match:")
    print(f"  {'Algorithm':<15} {'Environment':<20} {'Why'}")
    print("-" * 65)
    rows = [
        ("Q-learning",  "FrozenLake",  "Small discrete states → tabular Q-table fits"),
        ("SARSA",       "FrozenLake",  "Same — tabular, on-policy version"),
        ("DQN",         "CartPole",    "Continuous state → need network, not table"),
        ("REINFORCE",   "CartPole",    "Continuous state, policy gradient fits naturally"),
        ("PPO/SAC",     "MuJoCo/Robot","Continuous state AND continuous actions"),
    ]
    for alg, env_, why in rows:
        print(f"  {alg:<15} {env_:<20} {why}")

In [11]:
print("RL with Gymnasium — Two Examples")
print("Example 1: FrozenLake (discrete)  → Q-learning + SARSA")

# ── Example 1: FrozenLake ─────────────────────────────────────────────
print("\n" + "─"*60)
print("EXAMPLE 1 — FrozenLake-v1 (discrete states, tabular methods)")
print("─"*60)

q_table    = run_frozenlake_qlearning(n_episodes=5000)

# inference
run_gym_inference(q_table,     "FrozenLake-v1", "Q-learning",  n_episodes=5)

RL with Gymnasium — Two Examples
Example 1: FrozenLake (discrete)  → Q-learning + SARSA

────────────────────────────────────────────────────────────
EXAMPLE 1 — FrozenLake-v1 (discrete states, tabular methods)
────────────────────────────────────────────────────────────

EXAMPLE 1A: Q-Learning on FrozenLake-v1
States: 16, Actions: 4
Q-table size: 16 x 4 = 64 cells

Training complete (5000 episodes)
Success rate (last 100 episodes): 99%

Learned policy (best action per cell, 4x4 grid):
   DOWN  RIGHT   DOWN   LEFT
   DOWN   LEFT   DOWN   LEFT
  RIGHT   DOWN   DOWN   LEFT
   LEFT  RIGHT  RIGHT   LEFT

Q-table (showing max Q per state, 4x4 grid):
   0.77   0.81   0.86   0.81
   0.81   0.00   0.90   0.00
   0.86   0.90   0.95   0.00
   0.00   0.95   1.00   0.00

Key Gymnasium difference from hand-built:
  env.reset()  returns (state, info)  — not just state
  env.step()   returns (state, reward, terminated, truncated, info)
  done = terminated or truncated  — combine both flags
  Everythi

In [12]:
sarsa_table = run_frozenlake_sarsa(n_episodes=5000)
run_gym_inference(sarsa_table, "FrozenLake-v1", "SARSA",       n_episodes=5)


EXAMPLE 1B: SARSA on FrozenLake-v1
States: 16, Actions: 4

Training complete (5000 episodes)
Success rate (last 100 episodes): 98%

Learned policy (best action per cell, 4x4 grid):
   DOWN  RIGHT   DOWN  RIGHT
   DOWN   LEFT   DOWN   LEFT
  RIGHT   DOWN   DOWN   LEFT
   LEFT  RIGHT  RIGHT   LEFT

Key difference from Q-learning:
  next_action = choose_action(next_state)  <- epsilon-greedy picks a'
  next_q      = Q[next_state, next_action] <- uses a', not max
  Conservative: accounts for its own exploratory mistakes

--- Inference: SARSA on FrozenLake-v1 ---
  Scores over 5 episodes: [1, 1, 1, 1, 1]
  Average score: 1.0


In [13]:
# ── Example 2: CartPole ───────────────────────────────────────────────
print("\n" + "─"*60)
print("EXAMPLE 2 — CartPole-v1 (continuous states, neural networks)")
print("─"*60)

dqn_net       = run_cartpole_dqn(n_episodes=300)

# inference
if dqn_net:
    run_gym_inference(dqn_net,       "CartPole-v1", "DQN",       n_episodes=5)


────────────────────────────────────────────────────────────
EXAMPLE 2 — CartPole-v1 (continuous states, neural networks)
────────────────────────────────────────────────────────────

EXAMPLE 2A: DQN on CartPole-v1
State size: 4 continuous numbers
Actions: 2 (0=left, 1=right)
No Q-table possible — using neural network
  Episode   0 | Score    23 | Avg(last50)   23.0 | epsilon 0.99
  Episode  50 | Score    19 | Avg(last50)   31.1 | epsilon 0.77
  Episode 100 | Score   142 | Avg(last50)   55.5 | epsilon 0.60
  Episode 150 | Score   199 | Avg(last50)   77.6 | epsilon 0.47
  Episode 200 | Score   185 | Avg(last50)   80.1 | epsilon 0.37
  Episode 250 | Score   241 | Avg(last50)  120.4 | epsilon 0.28

Final avg score (last 50 episodes): 204.3
Score of 500 = perfect (pole balanced for max steps)

Key difference from 2x2 grid DQN:
  State is 4 continuous floats — fed directly to network (no encoding)
  Network is wider (64 units) to handle more complex state space
  Buffer is larger (10000) —

In [14]:
reinforce_net = run_cartpole_reinforce(n_episodes=500)


if reinforce_net:
    run_gym_inference(reinforce_net, "CartPole-v1", "REINFORCE", n_episodes=5)

# comparison summary
print_gym_comparison()



EXAMPLE 2B: REINFORCE on CartPole-v1
State size: 4 continuous numbers
Actions: 2 (0=left, 1=right)
  Episode   0 | Score    15 | Avg(last50)   15.0
  Episode  50 | Score    30 | Avg(last50)   27.0
  Episode 100 | Score    26 | Avg(last50)   25.1
  Episode 150 | Score    29 | Avg(last50)   33.6
  Episode 200 | Score    24 | Avg(last50)   33.1
  Episode 250 | Score    27 | Avg(last50)   47.9
  Episode 300 | Score    42 | Avg(last50)   64.8
  Episode 350 | Score   125 | Avg(last50)   89.5
  Episode 400 | Score   202 | Avg(last50)  152.3
  Episode 450 | Score    99 | Avg(last50)  167.2

Final avg score (last 50 episodes): 235.6
Score of 500 = perfect (pole balanced for max steps)

Key difference from 2x2 grid REINFORCE:
  State is 4 continuous floats — fed directly (no one-hot encoding)
  torch.distributions.Categorical used for cleaner sampling
  Returns normalized per episode (reduces variance for CartPole)
  Everything else (softmax output, G_t, no buffer) IDENTICAL

--- Inference: REI